In [1]:
!pip install chromadb sentence_transformers google-generativeai FlagEmbedding

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 8.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 14.6 MB/s eta 0:00:00
 

In [2]:
!pip install -U FlagEmbedding transformers==4.44.0 accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of flagembedding to determine which version is compatible with other requirements. This could take a while.
  Using cached FlagEmbedding-1.3.5-py3-none-any.whl
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 9.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.8/161.8 kB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.8/177.8 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is still looking at multiple versions of flagembedding to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.1/147.1 kB 8.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 95.3 MB/s eta 0:00:

In [ ]:
import time
import chromadb
# We use the specific classes for BGE-M3
from FlagEmbedding import BGEM3FlagModel, FlagReranker
import google.generativeai as genai
from google.api_core import exceptions

# --- 1. SETUP ---
GEMINI_API_KEY = ""
genai.configure(api_key=GEMINI_API_KEY)
model_llm = genai.GenerativeModel('gemini-2.0-flash')

# --- 2. HEAVY MODELS LOAD (Run once) ---
print("📥 Loading SOTA Models (this takes a minute)...")

# A. Embedding Model (BGE-M3) - The "Retriever"
# Supports French natively & Hybrid Search (Dense + Sparse)
embedding_model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

# B. Reranker Model (BGE-Reranker-V2-M3) - The "Judge"
# This acts like a mini-GPT-4 just for grading relevance
reranker_model = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)

# C. Vector DB Setup
chroma_client = chromadb.Client()

# ✅ FINAL CORRECTED CLASS
class BGEEmbeddingFunction:
    def __init__(self, model):
        self.model = model

    def __call__(self, input):
        return self.embed_documents(input)

    def embed_documents(self, input):
        # Handle list vs string input safely
        if isinstance(input, str):
            input = [input]
        # BGE-M3 specific extraction
        embeddings = self.model.encode(input)['dense_vecs']
        return embeddings.tolist()

    def embed_query(self, input):
        if isinstance(input, str):
            input = [input]
        embeddings = self.model.encode(input)['dense_vecs']
        return embeddings.tolist()

    # THIS WAS THE MISSING PART CAUSING YOUR ERROR
    def name(self):
        return "bge_m3_embedding"

# --- IMPORTANT: Use a NEW collection name to avoid conflicts ---
# If you use the old name 'french_docs', it might crash with a "Dimension Mismatch"
# because of your previous failed attempts.
collection = chroma_client.get_or_create_collection(
    name="french_docs_v2",
    embedding_function=BGEEmbeddingFunction(embedding_model)
)
# --- 3. CORE RETRIEVAL LOGIC ---

def retrieve_and_rerank(query, top_k=5):
    print(f"   🔎 Looking for: '{query}'")

    # Step 1: Broad Search (Vector DB)
    # We fetch more candidates (15) because we trust the Reranker to filter them
    results = collection.query(query_texts=[query], n_results=15)
    docs = results['documents'][0]

    if not docs: return []

    # Step 2: Cross-Encoder Reranking (The "Advanced" Part)
    # The reranker looks at the Query + Doc pairs and gives a score (-10 to +10)
    pairs = [[query, doc] for doc in docs]
    scores = reranker_model.compute_score(pairs)

    # Combine Doc + Score
    scored_docs = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)

    # Step 3: Threshold Filtering
    # If the reranker score is < 0, it means "Irrelevant". We discard it.
    final_docs = []
    print("      --- Reranking Report ---")
    for doc, score in scored_docs[:top_k]:
        status = "✅ Kept" if score > 0 else "❌ Low Score"
        print(f"      [{status}] Score: {score:.2f} | Text: {doc[:50]}...")
        if score > -2: # Lenient threshold
            final_docs.append(doc)

    return final_docs

# --- 4. THE AGENTIC LOOP (Same as before, but smarter) ---

def safe_ask_gemini(prompt):
    """
    Bulletproof caller that catches 429 errors by string analysis
    and waits aggressively (up to 2 minutes).
    """
    max_retries = 5
    for attempt in range(max_retries):
        try:
            # Try to generate
            return model_llm.generate_content(prompt).text.strip()

        except Exception as e:
            error_str = str(e)

            # Check if it's a Quota/Rate Limit error (429)
            if "429" in error_str or "Quota" in error_str or "ResourceExhausted" in error_str:
                # EXPONENTIAL BACKOFF: 20s, 40s, 60s, 80s, 100s
                wait_time = 20 * (attempt + 1)
                print(f"      🛑 Quota Hit (Attempt {attempt+1}/{max_retries}). Cooling down for {wait_time}s...")
                time.sleep(wait_time)
            else:
                # If it's a real error (like bad API key), crash immediately
                return f"Error: {error_str}"

    return "Error: Failed after 5 retries. Google is limiting you heavily right now."

def run_pipeline(user_query):
    # 1. Retrieve
    context_docs = retrieve_and_rerank(user_query)

    if not context_docs:
        return "Je n'ai trouvé aucune information pertinente dans les documents."

    context_str = "\n".join(context_docs)

    # 2. Generate (Force French Output)
    prompt = f"""
    Context (from Documents):
    {context_str}

    User Question:
    {user_query}

    Instructions:
    Answer the question strictly based on the context.
    If the context is technical, be precise.
    Output your answer in French.
    """

    # 3. Simple Hallucination Check
    # (We assume the Reranker did the heavy lifting, so we just generate)
    answer = safe_ask_gemini(prompt)
    return answer

# --- RUN IT ---
if __name__ == "__main__":
    # Install command: pip install FlagEmbedding

    # Example Ingestion (French)
    french_docs = [
        "Le produit net bancaire consolidé s'élève à 17,7 milliards de dirhams.",
        "Le résultat net part du groupe ressort à 5,9 milliards de dirhams.",
        "L'équipe a pris une pause café à 10h00.", # Trap document
        "La météo à Casablanca est ensoleillée."    # Trap document
    ]
    ids = [str(i) for i in range(len(french_docs))]

    print("📥 Indexing Documents...")
    collection.add(documents=french_docs, ids=ids)

    # Run Query
    q = "Quel est le résultat net part du groupe ?"
    print(f"\n🤖 REPONSE: {run_pipeline(q)}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


📥 Loading SOTA Models (this takes a minute)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

...

      🛑 Quota Hit (Attempt 1/5). Cooling down for 20s...


      🛑 Quota Hit (Attempt 2/5). Cooling down for 40s...


      🛑 Quota Hit (Attempt 3/5). Cooling down for 60s...


      🛑 Quota Hit (Attempt 4/5). Cooling down for 80s...


      🛑 Quota Hit (Attempt 5/5). Cooling down for 100s...

🤖 REPONSE: Error: Failed after 5 retries. Google is limiting you heavily right now.


In [5]:
!pip install huggingface_hub chromadb FlagEmbedding

In [ ]:
import time
import chromadb
from huggingface_hub import InferenceClient
from FlagEmbedding import BGEM3FlagModel, FlagReranker

# --- 1. CONFIGURATION ---
# Get your key: https://huggingface.co/settings/tokens
HF_API_KEY = ""

# MODEL: Qwen 2.5 72B (Best available on HF Free Tier)
# If this fails/timeouts, switch to "Qwen/Qwen2.5-7B-Instruct" (Faster/Dumber)
REPO_ID = "Qwen/Qwen2.5-72B-Instruct"

client = InferenceClient(api_key=HF_API_KEY)

# --- 2. LOCAL RETRIEVAL (Zero Cost, High Speed) ---
print("📥 Loading BGE-M3 (SOTA French Retrieval)...")
embedding_model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
reranker_model = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)

class BGEEmbeddingFunction:
    def __init__(self, model): self.model = model
    def __call__(self, input): return self.embed_documents(input)
    def embed_documents(self, input):
        if isinstance(input, str): input = [input]
        return self.model.encode(input)['dense_vecs'].tolist()
    def embed_query(self, input):
        if isinstance(input, str): input = [input]
        return self.model.encode(input)['dense_vecs'].tolist()
    def name(self): return "bge_m3_embedding"

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="hf_agent_docs",
    embedding_function=BGEEmbeddingFunction(embedding_model)
)

# --- 3. ROBUST HF CALLER (The Anti-Crash Layer) ---

def ask_hf(prompt, temperature=0.1):
    """
    Calls Hugging Face with auto-retries for 503/Loading errors.
    """
    retries = 3
    for attempt in range(retries):
        try:
            messages = [{"role": "user", "content": prompt}]
            response = client.chat_completion(
                model=REPO_ID,
                messages=messages,
                temperature=temperature,
                max_tokens=1024,
                stream=False
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            err = str(e)
            if "loading" in err.lower() or "503" in err:
                wait = 20 * (attempt + 1)
                print(f"      ⏳ Model loading/busy. Waiting {wait}s...")
                time.sleep(wait)
            elif "429" in err:
                print(f"      🛑 Rate limit. Waiting 60s...")
                time.sleep(60)
            else:
                return f"[Error: {err}]"
    return "[Fail: HF API is unresponsive]"

# --- 4. RETRIEVAL & RERANKING ---

def retrieve_and_rerank(query, top_k=3):
    print(f"   🔎 Searching for: '{query}'")

    # Broad Search
    results = collection.query(query_texts=[query], n_results=10)
    docs = results['documents'][0]
    if not docs: return []

    # Cross-Encoder Reranking
    pairs = [[query, doc] for doc in docs]
    scores = reranker_model.compute_score(pairs)

    # Sort
    scored_docs = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)

    final_docs = []
    print("      --- Reranker Report ---")
    for doc, score in scored_docs[:top_k]:
        status = "✅" if score > -2 else "❌"
        print(f"      [{status}] {score:.2f} | {doc[:40]}...")
        if score > -2: final_docs.append(doc)

    return final_docs

# --- 5. THE AGENTS (Qwen 72B) ---

def query_rewriter(original_query):
    prompt = f"""
    The search for '{original_query}' failed.
    Write a BETTER, keyword-focused search query in French.
    Output ONLY the query string.
    """
    print("      🔄 Agent: Rewriting Query...")
    return ask_hf(prompt)

def hallucination_checker(context, answer):
    prompt = f"""
    Context: {context}
    Answer: {answer}
    Task: Is the Answer supported by the Context?
    Output strictly 'YES' or 'NO'.
    """
    res = ask_hf(prompt).upper()
    return "YES" in res

def relevance_checker(query, answer):
    prompt = f"""
    User Query: {query}
    Generated Answer: {answer}
    Task: Does the Answer address the Query?
    Output strictly 'YES' or 'NO'.
    """
    res = ask_hf(prompt).upper()
    return "YES" in res

# --- 6. MAIN PIPELINE ---

def run_hf_agent(user_query):
    current_query = user_query

    for attempt in range(3): # Max 3 loops
        print(f"\n--- Loop {attempt+1} (Query: {current_query}) ---")

        # 1. Retrieve
        docs = retrieve_and_rerank(current_query)
        if not docs:
            print("   ⚠️ No docs found.")
            current_query = query_rewriter(current_query)
            continue

        context = "\n".join(docs)

        # 2. Generate
        prompt = f"""
        Tu es un analyste expert. Réponds à la question en utilisant le contexte.

        Contexte:
        {context}

        Question: {user_query}

        Réponse (en français):
        """
        answer = ask_hf(prompt)
        print(f"   📝 Draft Answer: {answer[:50]}...")

        # 3. Supervise
        if not hallucination_checker(context, answer):
            print("   👮 Hallucination Detected! (Fact Check Failed)")
            current_query = query_rewriter(current_query)
            continue

        if not relevance_checker(user_query, answer):
            print("   👮 Irrelevance Detected! (Wrong Topic)")
            current_query = query_rewriter(current_query)
            continue

        print("   ✅ Answer Approved.")
        return answer

    return "Échec: Je n'ai pas pu valider une réponse fiable."

# --- TEST ---
if __name__ == "__main__":
    # Load your Flattened PDF Text Here
    # (Just an example)
    sample_text = [
        "Le résultat net part du groupe est de 5,9 milliards de dirhams.",
        "Attijariwafa bank a tenu son conseil le 28 juillet 2025.",
        "Le temps est ensoleillé à Casablanca."
    ]
    ids = [str(i) for i in range(len(sample_text))]

    print("📥 Indexing Data...")
    collection.add(documents=sample_text, ids=ids)

    q = "Quel est le résultat net ?"
    print(f"\n🤖 Final Answer:\n{run_hf_agent(q)}")

📥 Loading BGE-M3 (SOTA French Retrieval)...


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

📥 Indexing Data...

--- Loop 1 (Query: Quel est le résultat net ?) ---
   🔎 Searching for: 'Quel est le résultat net ?'
      --- Reranker Report ---
      [✅] 0.28 | Le résultat net part du groupe est de 5,...
      [❌] -10.99 | Attijariwafa bank a tenu son conseil le ...
      [❌] -11.02 | Le temps est ensoleillé à Casablanca....
   📝 Draft Answer: Le résultat net part du groupe est de 5,9 milliard...
   ✅ Answer Approved.

🤖 Final Answer:
Le résultat net part du groupe est de 5,9 milliards de dirhams.


In [6]:
!pip install pdfplumber langchain langchain-community langchain-experimental chromadb sentence-transformers huggingface_hub FlagEmbedding accelerate

In [7]:
!pip install langchain-experimental
!pip install --force-reinstall --no-cache-dir langchain-experimental langchain-community langchain-core

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 197.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 234.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 209.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 139.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.5/500.5 kB 453.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 464.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 464.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.7/325.7 kB 420.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 363.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 357.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 433.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import time
import json
import pdfplumber
import chromadb
import re
import numpy as np
from typing import List, Dict, Any
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# NLP & AI Imports
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from huggingface_hub import InferenceClient
from FlagEmbedding import FlagReranker

# --- 1. CONFIGURATION ---
# Get your key: https://huggingface.co/settings/tokens
HF_API_KEY = ""

# Models
LLM_REPO_ID = "Qwen/Qwen2.5-72B-Instruct" # The Brain
EMBEDDING_MODEL_NAME = "BAAI/bge-m3"      # The Retriever (SOTA French)
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3" # The Judge

# --- 2. PRE-PROCESSING (Adapted from YOUR Notebook) ---

def extract_text_with_metadata(pdf_path: str, pages_to_ignore: int = 8) -> List[Document]:
    """
    Extracts text using pdfplumber, keeping page numbers intact.
    Returns a list of LangChain Document objects ready for chunking.
    """
    print(f"📄 Extracting text from {pdf_path} (Ignoring first {pages_to_ignore} pages)...")
    documents = []

    try:
        with pdfplumber.open(pdf_path) as pdf:
            # Slice pages
            pages_to_process = pdf.pages[pages_to_ignore:]

            for i, page in enumerate(pages_to_process):
                text = page.extract_text()
                if not text: continue

                # Calculate real page number
                real_page_num = pages_to_ignore + i + 1

                # Create standard Document object
                doc = Document(
                    page_content=text,
                    metadata={"source": pdf_path, "page": real_page_num}
                )
                documents.append(doc)

        print(f"   ✅ Extracted {len(documents)} pages.")
        return documents
    except Exception as e:
        print(f"   ❌ Error reading PDF: {e}")
        return []

# --- 3. SEMANTIC CHUNKING (The "Custom" Upgrade) ---
# REPLACED: No longer relies on langchain_experimental
def apply_semantic_chunking(documents: List[Document]):
    """
    Splits text based on MEANING using Cosine Similarity.
    Manual implementation to bypass import errors.
    """
    print("🧠 Starting Semantic Chunking (Custom Implementation)...")

    # Load the model directly for mathematical operations
    model = SentenceTransformer(EMBEDDING_MODEL_NAME)

    final_chunks = []

    for doc in documents:
        # 1. Split into sentences (Regex for robustness)
        # Splits on '.', '?', '!' followed by whitespace
        sentences = re.split(r'(?<=[.?!])\s+', doc.page_content)

        # Filter empty sentences
        sentences = [s.strip() for s in sentences if len(s.strip()) > 5]

        if len(sentences) < 2:
            final_chunks.append(doc)
            continue

        # 2. Embed sentences
        embeddings = model.encode(sentences)

        # 3. Calculate Cosine Distance between adjacent sentences
        distances = []
        for i in range(len(embeddings) - 1):
            # Calculate similarity
            sim = cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0]
            distances.append(sim)

        # 4. Determine Breakpoints
        # We split at the 20% lowest similarities (natural topic shifts)
        if not distances:
            final_chunks.append(doc)
            continue

        threshold = np.percentile(distances, 20)

        # 5. Group sentences into Chunks
        current_chunk_sentences = [sentences[0]]

        for i, dist in enumerate(distances):
            if dist < threshold: # Low similarity -> Topic Change -> Split
                text = " ".join(current_chunk_sentences)
                # Save chunk as Document to match your existing code structure
                final_chunks.append(Document(page_content=text, metadata=doc.metadata))
                current_chunk_sentences = [sentences[i+1]]
            else:
                current_chunk_sentences.append(sentences[i+1])

        # Add the last pending chunk
        if current_chunk_sentences:
            text = " ".join(current_chunk_sentences)
            final_chunks.append(Document(page_content=text, metadata=doc.metadata))

    print(f"   ✅ Created {len(final_chunks)} semantic chunks.")
    return final_chunks

# --- 4. VECTOR DATABASE SETUP ---

# Custom wrapper for ChromaDB to use our BGE-M3 model locally
class BGEEmbeddingFunction:
    def __init__(self, model_name):
        self.embedding_model = HuggingFaceEmbeddings(model_name=model_name)

    def __call__(self, input):
        return self.embed_documents(input)

    def embed_documents(self, input):
        # Ensure input is a list
        if isinstance(input, str):
            input = [input]
        return self.embedding_model.embed_documents(input)

    def embed_query(self, input):
        # FIX: Handle case where Chroma sends a list for a query
        if isinstance(input, list):
            return self.embedding_model.embed_documents(input)
        return self.embedding_model.embed_query(input)

    def name(self):
        return "bge_m3_semantic"

print("📥 Loading Reranker & DB...")
reranker = FlagReranker(RERANKER_MODEL_NAME, use_fp16=True)

# Use PersistentClient to save data locally
chroma_client = chromadb.PersistentClient(path="./my_local_db")

# Create Collection
collection = chroma_client.get_or_create_collection(
    name="labor_code_semantic",
    embedding_function=BGEEmbeddingFunction(EMBEDDING_MODEL_NAME)
)

# --- 5. AGENTIC CORE (The Intelligence) ---

client = InferenceClient(api_key=HF_API_KEY)

def ask_hf(prompt, system_role="You are a helpful assistant."):
    """Robust HF Caller with Retry Logic"""
    for attempt in range(3):
        try:
            messages = [
                {"role": "system", "content": system_role},
                {"role": "user", "content": prompt}
            ]
            response = client.chat_completion(
                model=LLM_REPO_ID,
                messages=messages,
                temperature=0.1,
                max_tokens=1024
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if "loading" in str(e).lower() or "503" in str(e):
                print(f"      ⏳ Model loading... waiting {20*(attempt+1)}s")
                time.sleep(20 * (attempt + 1))
            else:
                print(f"      ❌ API Error: {e}")
                break
    return "Error: Model unavailable."

# --- 6. RETRIEVAL PIPELINE (Retrieve -> Rerank) ---

def retrieve_advanced(query, top_k=5):
    print(f"   🔎 Step 1: Broad Search for '{query}'...")

    # 1. Broad Vector Search
    results = collection.query(query_texts=[query], n_results=15)
    docs = results['documents'][0]
    metadatas = results['metadatas'][0]

    if not docs: return []

    # 2. Reranking
    print("   ⚖️ Step 2: Reranking Candidates...")
    pairs = [[query, doc] for doc in docs]
    scores = reranker.compute_score(pairs)

    # Combine Data
    ranked_results = []
    for doc, meta, score in zip(docs, metadatas, scores):
        ranked_results.append({"text": doc, "meta": meta, "score": score})

    # Sort by Score
    ranked_results.sort(key=lambda x: x["score"], reverse=True)

    # Filter Top K (with simple threshold)
    final_context = []
    for item in ranked_results[:top_k]:
        if item["score"] > -2: # Threshold for "relevance"
            citation = f"[Page {item['meta']['page']}]"
            final_context.append(f"{citation} {item['text']}")

    return final_context

# --- 7. THE AGENTS ---

def query_rewriter(query):
    prompt = f"""
    The user asked: "{query}"
    This search query failed to find relevant legal documents.
    Write a BETTER, keyword-rich search query in French focused on Labor Law terminology.
    Output ONLY the new query.
    """
    print("      🔄 Agent: Rewriting Query...")
    return ask_hf(prompt)

def supervisor_agent(context, answer, query):
    """Checks for Hallucinations AND Relevance"""
    prompt = f"""
    Context: {context}
    User Query: {query}
    Proposed Answer: {answer}

    Task: Validate this answer.
    1. Is the answer supported by the Context?
    2. Does it directly answer the User Query?

    Output strictly 'YES' if it passes both. Output 'NO' if it fails either.
    """
    res = ask_hf(prompt).upper()
    return "YES" in res

# --- 8. MAIN WORKFLOW ---

def run_agentic_rag(user_query):
    current_query = user_query

    for attempt in range(3): # Max 3 loops
        print(f"\n🚀 --- Loop {attempt+1}: '{current_query}' ---")

        # A. Retrieve
        context_list = retrieve_advanced(current_query)
        if not context_list:
            print("   ⚠️ No relevant documents found.")
            current_query = query_rewriter(current_query)
            continue

        context_str = "\n\n".join(context_list)

        # B. Generate
        print("   ✍️ Generating Answer...")
        system_prompt = (
            "Tu es un expert en Droit du Travail Marocain. "
            "Réponds à la question en te basant UNIQUEMENT sur le contexte fourni. "
            "Cite toujours le numéro de page indiqué dans le contexte (ex: [Page 12]). "
            "Si la réponse n'est pas dans le contexte, dis-le clairement."
        )
        user_prompt = f"Contexte:\n{context_str}\n\nQuestion: {user_query}"

        answer = ask_hf(user_prompt, system_role=system_prompt)

        # C. Supervise
        print("   👮 Supervisor checking answer...")
        if supervisor_agent(context_str, answer, user_query):
            print("   ✅ Answer Validated.")
            return answer
        else:
            print("   ❌ Answer Rejected (Hallucination or Irrelevant).")
            current_query = query_rewriter(current_query)
            continue

    return "Je n'ai pas trouvé de réponse fiable dans le document."

# --- EXECUTION ---
if __name__ == "__main__":

    # 1. Load Data (Run this only once)
    pdf_name = "Code-du-Travail-Maroc.pdf" # Make sure this file exists!

    # Check if DB is empty
    if collection.count() == 0:
        raw_docs = extract_text_with_metadata(pdf_name, pages_to_ignore=8)
        if raw_docs:
            semantic_chunks = apply_semantic_chunking(raw_docs)

            # Add to Chroma
            ids = [f"id_{i}" for i in range(len(semantic_chunks))]
            # We iterate over the Document objects returned by our custom function
            texts = [c.page_content for c in semantic_chunks]
            metadatas = [c.metadata for c in semantic_chunks]

            print("📥 Indexing to ChromaDB...")
            collection.add(documents=texts, metadatas=metadatas, ids=ids)
    else:
        print("📥 Database already loaded. Skipping ingestion.")

    # 2. Run Query
    q = "Quel est le délai de préavis pour une démission ?"
    print(f"\n🤖 FINAL ANSWER:\n{run_agentic_rag(q)}")

📥 Loading Reranker & DB...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/tmp/ipython-input-3422783791.py:131: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embedding_model = HuggingFaceEmbeddings(model_name=model_name)


📄 Extracting text from Code-du-Travail-Maroc.pdf (Ignoring first 8 pages)...
   ✅ Extracted 121 pages.
🧠 Starting Semantic Chunking (Custom Implementation)...
   ✅ Created 371 semantic chunks.
📥 Indexing to ChromaDB...

🚀 --- Loop 1: 'Quel est le délai de préavis pour une démission ?' ---
   🔎 Step 1: Broad Search for 'Quel est le délai de préavis pour une démission ?'...
   ⚖️ Step 2: Reranking Candidates...
   ✍️ Generating Answer...
   👮 Supervisor checking answer...
   ✅ Answer Validated.

🤖 FINAL ANSWER:
Le délai de préavis pour une démission n'est pas spécifiquement mentionné dans le contexte fourni. Cependant, selon l'article 43 [Page 23], le délai de préavis est généralement réglementé par les textes législatifs et réglementaires, le contrat de travail, la convention collective de travail, le règlement intérieur ou les usages. Il est également indiqué que toute clause fixant le délai de préavis à moins de huit jours est nulle. Par conséquent, le délai de préavis pour une démiss

In [ ]:
import time

# 1. Définition des questions de test (classées par niveau)
test_questions = [
    # --- NIVEAU 1 : Récupération Factuelle ---
    "Quelle est la durée normale de travail hebdomadaire dans les activités non agricoles ?",
    "Quel est l'âge minimum pour être admis au travail ?",
    "Quelle est la durée du congé de maternité ?",

    # --- NIVEAU 2 : Distinctions & Nuances ---
    "Quelle est la période d'essai pour un cadre en CDI ?",
    "Combien de jours d'absence sont autorisés en cas de décès d'un conjoint ?",
    "Peut-on conclure un CDD pour le lancement d'un nouveau produit ? Si oui, pour quelle durée ?",

    # --- NIVEAU 3 : Procédures Complexes ---
    "Quelles sont les étapes à suivre pour licencier un salarié pour faute grave ?",
    "Est-il permis de faire travailler des femmes la nuit ?",

    # --- NIVEAU 4 : Pièges & Limites (Test d'hallucination) ---
    "Quel est le montant exact du SMIG actuellement ?",  # Doit dire qu'il est fixé par voie réglementaire (pas de chiffre dans le PDF)
    "Un employeur peut-il payer ses salariés le jour de leur repos hebdomadaire ?"
]

# 2. Boucle d'exécution
print(f"🚀 Démarrage du benchmark sur {len(test_questions)} questions...\n")

results = []

for i, question in enumerate(test_questions, 1):
    print(f"🔹 [Question {i}/{len(test_questions)}] : {question}")

    start_time = time.time()

    # Appel de votre agent RAG
    try:
        response = run_agentic_rag(question)
    except Exception as e:
        response = f"❌ ERREUR : {str(e)}"

    duration = time.time() - start_time

    print(f"🤖 Réponse ({duration:.1f}s) :\n{response}")
    print("-" * 60 + "\n")

    # Stockage pour analyse ultérieure si besoin
    results.append({
        "question": question,
        "response": response,
        "duration": duration
    })

    # Petite pause pour éviter de surcharger l'API Hugging Face
    time.sleep(2)

print("✅ Test terminé.")

🚀 Démarrage du benchmark sur 10 questions...

🔹 [Question 1/10] : Quelle est la durée normale de travail hebdomadaire dans les activités non agricoles ?

🚀 --- Loop 1: 'Quelle est la durée normale de travail hebdomadaire dans les activités non agricoles ?' ---
   🔎 Step 1: Broad Search for 'Quelle est la durée normale de travail hebdomadaire dans les activités non agricoles ?'...
   ⚖️ Step 2: Reranking Candidates...
   ✍️ Generating Answer...
   👮 Supervisor checking answer...
   ✅ Answer Validated.
🤖 Réponse (4.5s) :
La durée normale de travail hebdomadaire dans les activités non agricoles est fixée à 44 heures par semaine [Page 51].
------------------------------------------------------------

🔹 [Question 2/10] : Quel est l'âge minimum pour être admis au travail ?

🚀 --- Loop 1: 'Quel est l'âge minimum pour être admis au travail ?' ---
   🔎 Step 1: Broad Search for 'Quel est l'âge minimum pour être admis au travail ?'...
   ⚖️ Step 2: Reranking Candidates...
   ✍️ Generating Answer.

In [ ]:
questions_complexes = [
    "Un salarié commet une insulte grave envers son employeur. Quelles sont les conséquences sur ses indemnités de préavis et de licenciement ?",
    "Si un salarié démissionne parce que son employeur l'a insulté gravement, peut-il prétendre à des indemnités ?",
    "Quelles périodes de suspension du contrat de travail sont comptabilisées pour le calcul de l'indemnité de licenciement ?",
    "Un salarié a 12 ans d'ancienneté. Quel est le taux de sa prime d'ancienneté et le taux horaire de son indemnité de licenciement pour sa 12ème année ?",
    "Un salarié devenu handicapé peut-il être licencié s'il ne peut plus faire son travail initial ?",
    "Peut-on notifier un licenciement pour faute grave à une femme pendant son congé de maternité ?",
    "Comment prouver un contrat de travail non écrit et quels documents l'employeur doit-il obligatoirement fournir ?"
]

for q in questions_complexes:
    print(f"\n🧐 TEST COMPLEXE : {q}")
    print(run_agentic_rag(q))


🧐 TEST COMPLEXE : Un salarié commet une insulte grave envers son employeur. Quelles sont les conséquences sur ses indemnités de préavis et de licenciement ?

🚀 --- Loop 1: 'Un salarié commet une insulte grave envers son employeur. Quelles sont les conséquences sur ses indemnités de préavis et de licenciement ?' ---
   🔎 Step 1: Broad Search for 'Un salarié commet une insulte grave envers son employeur. Quelles sont les conséquences sur ses indemnités de préavis et de licenciement ?'...
   ⚖️ Step 2: Reranking Candidates...
   ✍️ Generating Answer...
   👮 Supervisor checking answer...
   ✅ Answer Validated.
Selon le contexte fourni, si un salarié commet une insulte grave envers son employeur, cela constitue une faute grave [Page 22]. En cas de faute grave, le salarié peut être licencié sans préavis ni indemnité ni versement de dommages-intérêts [Page 26, Article 61]. Par conséquent, le salarié ne bénéficiera ni de l'indemnité de préavis ni de l'indemnité de licenciement.

🧐 TEST COMPLE

In [ ]:
!zip -r my_local_db.zip /content/'my_local_db'

  adding: content/my_local_db/ (stored 0%)
  adding: content/my_local_db/ec3f4698-886f-4a41-b4f5-03b278b1cb28/ (stored 0%)
  adding: content/my_local_db/ec3f4698-886f-4a41-b4f5-03b278b1cb28/header.bin (deflated 63%)
  adding: content/my_local_db/ec3f4698-886f-4a41-b4f5-03b278b1cb28/length.bin (deflated 63%)
  adding: content/my_local_db/ec3f4698-886f-4a41-b4f5-03b278b1cb28/data_level0.bin (deflated 100%)
  adding: content/my_local_db/ec3f4698-886f-4a41-b4f5-03b278b1cb28/link_lists.bin (stored 0%)
  adding: content/my_local_db/chroma.sqlite3 (deflated 43%)


In [ ]:
from huggingface_hub import snapshot_download

# Download the model to a specific local folder
model_path = snapshot_download(
    repo_id="BAAI/bge-reranker-v2-m3", # Or whichever model you use
    local_dir="models/reranker",
    local_dir_use_symlinks=False
)
print(f"Model downloaded to: {model_path}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Model downloaded to: /content/models/reranker


In [ ]:
!zip -r models.zip /content/'models'

  adding: content/models/ (stored 0%)
  adding: content/models/reranker/ (stored 0%)
  adding: content/models/reranker/config.json (deflated 48%)
  adding: content/models/reranker/special_tokens_map.json (deflated 85%)
  adding: content/models/reranker/model.safetensors (deflated 42%)
  adding: content/models/reranker/assets/ (stored 0%)
  adding: content/models/reranker/assets/llama-index.png (deflated 6%)
  adding: content/models/reranker/assets/miracl-bge-m3.png (deflated 9%)
  adding: content/models/reranker/assets/BEIR-bge-en-v1.5.png (deflated 7%)
  adding: content/models/reranker/assets/BEIR-e5-mistral.png (deflated 6%)
  adding: content/models/reranker/assets/CMTEB-retrieval-bge-zh-v1.5.png (deflated 5%)
  adding: content/models/reranker/sentencepiece.bpe.model (deflated 49%)
  adding: content/models/reranker/tokenizer.json (deflated 76%)
  adding: content/models/reranker/tokenizer_config.json (deflated 76%)
  adding: content/models/reranker/.gitattributes (deflated 87%)
  addi

In [2]:
import os
import time
import json
import pdfplumber
import chromadb
import re
import numpy as np
from typing import List, Dict, Any
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# NLP & AI Imports
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from huggingface_hub import InferenceClient
from FlagEmbedding import FlagReranker

EMBEDDING_MODEL_NAME = "BAAI/bge-m3"      # The Retriever (SOTA French)


def extract_text_with_metadata(pdf_path: str, pages_to_ignore: int = 8) -> List[Document]:
    """
    Extracts text using pdfplumber, keeping page numbers intact.
    Returns a list of LangChain Document objects ready for chunking.
    """
    print(f"📄 Extracting text from {pdf_path} (Ignoring first {pages_to_ignore} pages)...")
    documents = []

    try:
        with pdfplumber.open(pdf_path) as pdf:
            # Slice pages
            pages_to_process = pdf.pages[pages_to_ignore:]

            for i, page in enumerate(pages_to_process):
                text = page.extract_text()
                if not text: continue

                # Calculate real page number
                real_page_num = pages_to_ignore + i + 1

                # Create standard Document object
                doc = Document(
                    page_content=text,
                    metadata={"source": pdf_path, "page": real_page_num}
                )
                documents.append(doc)

        print(f"   ✅ Extracted {len(documents)} pages.")
        return documents
    except Exception as e:
        print(f"   ❌ Error reading PDF: {e}")
        return []

# --- 3. SEMANTIC CHUNKING (The "Custom" Upgrade) ---
# REPLACED: No longer relies on langchain_experimental
def apply_semantic_chunking(documents: List[Document]):
    """
    Splits text based on MEANING using Cosine Similarity.
    Manual implementation to bypass import errors.
    """
    print("🧠 Starting Semantic Chunking (Custom Implementation)...")

    # Load the model directly for mathematical operations
    model = SentenceTransformer(EMBEDDING_MODEL_NAME)

    final_chunks = []

    for doc in documents:
        # 1. Split into sentences (Regex for robustness)
        # Splits on '.', '?', '!' followed by whitespace
        sentences = re.split(r'(?<=[.?!])\s+', doc.page_content)

        # Filter empty sentences
        sentences = [s.strip() for s in sentences if len(s.strip()) > 5]

        if len(sentences) < 2:
            final_chunks.append(doc)
            continue

        # 2. Embed sentences
        embeddings = model.encode(sentences)

        # 3. Calculate Cosine Distance between adjacent sentences
        distances = []
        for i in range(len(embeddings) - 1):
            # Calculate similarity
            sim = cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0]
            distances.append(sim)

        # 4. Determine Breakpoints
        # We split at the 20% lowest similarities (natural topic shifts)
        if not distances:
            final_chunks.append(doc)
            continue

        threshold = np.percentile(distances, 20)

        # 5. Group sentences into Chunks
        current_chunk_sentences = [sentences[0]]

        for i, dist in enumerate(distances):
            if dist < threshold: # Low similarity -> Topic Change -> Split
                text = " ".join(current_chunk_sentences)
                # Save chunk as Document to match your existing code structure
                final_chunks.append(Document(page_content=text, metadata=doc.metadata))
                current_chunk_sentences = [sentences[i+1]]
            else:
                current_chunk_sentences.append(sentences[i+1])

        # Add the last pending chunk
        if current_chunk_sentences:
            text = " ".join(current_chunk_sentences)
            final_chunks.append(Document(page_content=text, metadata=doc.metadata))

    print(f"   ✅ Created {len(final_chunks)} semantic chunks.")
    return final_chunks

# --- 4. VECTOR DATABASE SETUP ---

# Custom wrapper for ChromaDB to use our BGE-M3 model locally
class BGEEmbeddingFunction:
    def __init__(self, model_name):
        self.embedding_model = HuggingFaceEmbeddings(model_name=model_name)

    def __call__(self, input):
        return self.embed_documents(input)

    def embed_documents(self, input):
        # Ensure input is a list
        if isinstance(input, str):
            input = [input]
        return self.embedding_model.embed_documents(input)

    def embed_query(self, input):
        # FIX: Handle case where Chroma sends a list for a query
        if isinstance(input, list):
            return self.embedding_model.embed_documents(input)
        return self.embedding_model.embed_query(input)

    def name(self):
        return "bge_m3_semantic"



# Use PersistentClient to save data locally
chroma_client = chromadb.PersistentClient(path="./my_local_db")

# Create Collection
collection = chroma_client.get_or_create_collection(
    name="labor_code_semantic",
    embedding_function=BGEEmbeddingFunction(EMBEDDING_MODEL_NAME)
)


# --- EXECUTION ---
if __name__ == "__main__":
    # 1. Liste des fichiers avec leurs pages à ignorer respectives
    pdf_files = {
        'Code-du-Travail-Maroc.pdf' : 8,
        'Maroc-Decrets-code-travail.pdf' : 1,
        'code-accident-travail-maroc.pdf' : 1,
        'dahir_1-72-184.pdf': 0,
        'heures-supplementaires.pdf': 0,
        'maroc-code-couverture-medicale.pdf' : 1,
        'nc-cnss.pdf' : 5,
        'Réussire sa retraite.pdf' : 3,
        'Guide Ramedistes_fr_01_12def(2)-1-8.pdf' : 3
    }

    # 2. Vérification des fichiers déjà présents pour éviter les doublons
    existing_files = []
    if collection.count() > 0:
        # On récupère les sources déjà indexées dans la base
        existing_metas = collection.get(include=['metadatas'])['metadatas']
        existing_files = set([m['source'] for m in existing_metas])
        print(f"📥 Base déjà chargée avec {len(existing_files)} fichiers.")
    else:
        print("📥 Base vide. Début de l'ingestion totale.")

    #

    # 3. Boucle de traitement
    for pdf_name, pages_to_ignore in pdf_files.items():

        # Vérifier si le fichier a déjà été traité
        if pdf_name in existing_files:
            print(f"⏩ {pdf_name} est déjà indexé. Sauter.")
            continue

        # Vérifier si le fichier existe physiquement
        if not os.path.exists(pdf_name):
            print(f"⚠️ Fichier {pdf_name} introuvable. Vérifiez le nom ou le chemin.")
            continue

        print(f"\n📄 Traitement de : {pdf_name}...")

        # A. Extraction
        raw_docs = extract_text_with_metadata(pdf_name, pages_to_ignore=pages_to_ignore)

        if raw_docs:
            # B. Chunking sémantique
            semantic_chunks = apply_semantic_chunking(raw_docs)

            # C. Préparation pour ChromaDB
            # On crée un ID unique combinant le nom du fichier et l'index du chunk
            ids = [f"{pdf_name}_chunk_{i}" for i in range(len(semantic_chunks))]
            texts = [c.page_content for c in semantic_chunks]
            metadatas = [c.metadata for c in semantic_chunks]

            print(f"📥 Indexation de {len(semantic_chunks)} segments dans ChromaDB...")
            collection.add(documents=texts, metadatas=metadatas, ids=ids)
        else:
            print(f"❌ Impossible d'extraire du texte de {pdf_name}")

    print(f"\n✅ Terminé ! Total de segments en base : {collection.count()}")




/tmp/ipython-input-1172297242.py:123: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embedding_model = HuggingFaceEmbeddings(model_name=model_name)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

📥 Base vide. Début de l'ingestion totale.

📄 Traitement de : Code-du-Travail-Maroc.pdf...
📄 Extracting text from Code-du-Travail-Maroc.pdf (Ignoring first 8 pages)...
   ✅ Extracted 121 pages.
🧠 Starting Semantic Chunking (Custom Implementation)...
   ✅ Created 371 semantic chunks.
📥 Indexation de 371 segments dans ChromaDB...

📄 Traitement de : Maroc-Decrets-code-travail.pdf...
📄 Extracting text from Maroc-Decrets-code-travail.pdf (Ignoring first 1 pages)...
   ✅ Extracted 34 pages.
🧠 Starting Semantic Chunking (Custom Implementation)...
   ✅ Created 81 semantic chunks.
📥 Indexation de 81 segments dans ChromaDB...

📄 Traitement de : code-accident-travail-maroc.pdf...
📄 Extracting text from code-accident-travail-maroc.pdf (Ignoring first 1 pages)...
   ✅ Extracted 58 pages.
🧠 Starting Semantic Chunking (Custom Implementation)...
   ✅ Created 166 semantic chunks.
📥 Indexation de 166 segments dans ChromaDB...

📄 Traitement de : dahir_1-72-184.pdf...
📄 Extracting text from dahir_1-72-184.

In [3]:
!zip -r my_local_db.zip /content/'my_local_db'

  adding: content/my_local_db/ (stored 0%)
  adding: content/my_local_db/chroma.sqlite3 (deflated 40%)
  adding: content/my_local_db/a1a1a9be-294e-4fe5-ab44-4319f2caff12/ (stored 0%)
  adding: content/my_local_db/a1a1a9be-294e-4fe5-ab44-4319f2caff12/header.bin (deflated 63%)
  adding: content/my_local_db/a1a1a9be-294e-4fe5-ab44-4319f2caff12/length.bin (deflated 61%)
  adding: content/my_local_db/a1a1a9be-294e-4fe5-ab44-4319f2caff12/data_level0.bin (deflated 100%)
  adding: content/my_local_db/a1a1a9be-294e-4fe5-ab44-4319f2caff12/link_lists.bin (stored 0%)
